# Network Catchment — Stage 1 (geometry only)

Click a point on the map, set a network distance, and the tool pulls the walkable street network from OSM, computes what's **reachable along the network** within that distance, and displays:

- the reachable **road edges** (blue)
- the **network polygon footprint** (buffered catchment, blue fill)
- the naive **buffer circle** (faint red) for comparison

No attribute data is pulled yet — a later cell will retrieve data *into* this polygon.

> Requires live internet (OSM/Overpass).

In [ ]:
!pip install osmnx ipyleaflet geopandas networkx shapely rasterio

In [ ]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
from shapely.geometry import Point, mapping
from ipyleaflet import Map, Marker, Circle, GeoJSON, LayerGroup, basemaps
import ipywidgets as widgets
from IPython.display import display

ox.settings.use_cache = True          # cache successful fetches: 10 benchmark runs reuse them
ox.settings.log_console = False

# generous timeout on the DEFAULT Overpass endpoint (no mirror pin).
# The bbox-based feature fetch in the brief cell is what prevents the timeouts.
ox.settings.overpass_settings = '[out:json][timeout:180]'
ox.settings.requests_timeout = 180
ox.settings.overpass_rate_limit = True


In [ ]:
def network_catchment(lat, lon, dist_m, block_depth=40):
    """Edge-based network catchment, computed entirely in a metric CRS.
    The whole graph is projected to UTM up front, so every edge length, every
    Dijkstra distance, and every geometry operation is in real metres. Partially
    reachable street segments are cut at the exact remaining walking budget.

    Returns:
      edges_4326   - reachable street segments, cut to budget (EPSG:4326)
      filled_4326  - block-depth filled catchment for ALL aggregation (EPSG:4326)
      road_len     - total reachable street length (cut), metres
    """
    import math
    import geopandas as gpd
    from shapely.geometry import LineString, Point
    from shapely.ops import substring

    # fetch generously so the catchment is never clipped by the download boundary
    fetch_dist = int(dist_m * 3 + 500)
    G = ox.graph_from_point((lat, lon), dist=fetch_dist, network_type='walk', simplify=True)

    # project the ENTIRE graph to metric CRS up front: node x/y and geometry in metres
    G = ox.project_graph(G)
    UG = G.to_undirected()
    crs = UG.graph['crs']

    # guarantee every edge has a correct METRIC length (never trust a missing/degree length)
    for u, v, data in UG.edges(data=True):
        geom = data.get('geometry')
        if geom is not None:
            data['length'] = geom.length
        else:
            x1, y1 = UG.nodes[u]['x'], UG.nodes[u]['y']
            x2, y2 = UG.nodes[v]['x'], UG.nodes[v]['y']
            data['length'] = math.hypot(x2 - x1, y2 - y1)

    # snap the clicked point: project it into the graph CRS first
    pt = gpd.GeoSeries([Point(lon, lat)], crs=4326).to_crs(crs).iloc[0]
    center = ox.distance.nearest_nodes(UG, X=pt.x, Y=pt.y)

    # walking distance to every node (metric), capped at the budget
    lengths = nx.single_source_dijkstra_path_length(UG, center, cutoff=dist_m, weight='length')

    def edge_geometry(u, v, data):
        g = data.get('geometry')
        if g is not None:
            return g
        return LineString([(UG.nodes[u]['x'], UG.nodes[u]['y']),
                           (UG.nodes[v]['x'], UG.nodes[v]['y'])])

    def aligned(geom, px, py):
        x0, y0 = geom.coords[0]
        if (round(x0, 3), round(y0, 3)) != (round(px, 3), round(py, 3)):
            return LineString(list(geom.coords)[::-1])
        return geom

    kept = []
    for u, v, data in UG.edges(data=True):
        du, dv = lengths.get(u), lengths.get(v)
        if du is None and dv is None:
            continue
        geom = edge_geometry(u, v, data)
        L = data['length']
        if du is not None and dv is not None:
            kept.append(geom)
        elif du is not None:
            kept.append(substring(aligned(geom, UG.nodes[u]['x'], UG.nodes[u]['y']), 0, min(dist_m - du, L)))
        else:
            kept.append(substring(aligned(geom, UG.nodes[v]['x'], UG.nodes[v]['y']), 0, min(dist_m - dv, L)))

    edges_m = gpd.GeoDataFrame(geometry=kept, crs=crs)
    road_len = float(edges_m.length.sum())

    filled_m = edges_m.buffer(block_depth).union_all().buffer(-block_depth * 0.25)
    filled_4326 = gpd.GeoSeries([filled_m], crs=crs).to_crs(4326).iloc[0]

    return edges_m.to_crs(4326), filled_4326, road_len


In [ ]:
DEFAULT = (48.8602, 2.3550)  # Paris, near Le Marais / Rambuteau

m = Map(center=DEFAULT, zoom=16, basemap=basemaps.CartoDB.Positron, scroll_wheel_zoom=True)
marker = Marker(location=DEFAULT, draggable=True)
m.add(marker)
result_layer = LayerGroup()
m.add(result_layer)

dist_slider = widgets.IntSlider(value=150, min=50, max=1000, step=25, description='Network m:')
depth_slider = widgets.IntSlider(value=40, min=10, max=80, step=5, description='Block depth m:',
                                 style={'description_width': 'initial'})
run_btn = widgets.Button(description='Show network', button_style='primary')

# manual coordinate entry
lat_input = widgets.FloatText(value=DEFAULT[0], description='Lat:', layout={'width': '180px'})
lon_input = widgets.FloatText(value=DEFAULT[1], description='Lon:', layout={'width': '180px'})
goto_btn = widgets.Button(description='Go to point')

out = widgets.Output()
state = {'lat': DEFAULT[0], 'lon': DEFAULT[1]}

def _set_point(lat, lon, recenter=False):
    state['lat'], state['lon'] = lat, lon
    marker.location = (lat, lon)
    lat_input.value, lon_input.value = lat, lon
    if recenter:
        m.center = (lat, lon)

def on_click(**kwargs):
    if kwargs.get('type') == 'click':
        la, lo = kwargs['coordinates']
        _set_point(la, lo)

m.on_interaction(on_click)
marker.observe(lambda c: _set_point(marker.location[0], marker.location[1]), 'location')

def on_goto(_):
    _set_point(lat_input.value, lon_input.value, recenter=True)

goto_btn.on_click(on_goto)

def run(_):
    out.clear_output()
    result_layer.clear_layers()
    lat, lon, d = state['lat'], state['lon'], dist_slider.value
    bd = depth_slider.value
    with out:
        print(f'Pulling network @ ({lat:.5f}, {lon:.5f}), {d} m, block depth {bd} m ...')
    try:
        edges, filled, road_len = network_catchment(lat, lon, d, block_depth=bd)
    except Exception as e:
        with out:
            print('ERROR:', e)
        return

    result_layer.add_layer(Circle(location=(lat, lon), radius=d,
                                  color='#cc4444', fill_color='#cc4444',
                                  fill_opacity=0.05, weight=1))
    result_layer.add_layer(GeoJSON(
        data=mapping(filled),
        style={'color': '#1f6fb4', 'fillColor': '#9ec5e8', 'fillOpacity': 0.30, 'weight': 1}))
    result_layer.add_layer(GeoJSON(
        data=edges.__geo_interface__,
        style={'color': '#1f6fb4', 'weight': 3, 'opacity': 0.9}))

    with out:
        print(f'reachable road length: {road_len:.0f} m')
        print('filled catchment ready \u2014 available as `catchment_polygon` for the next cell.')

    globals()['catchment_polygon'] = filled
    globals()['catchment_edges'] = edges
    globals()['catchment_road_length_m'] = road_len
    globals()['click_point'] = (lat, lon)
    globals()['catchment_settings'] = {'distance_m': d, 'block_depth_m': bd}

run_btn.on_click(run)
display(widgets.VBox([
    widgets.HBox([dist_slider, depth_slider, run_btn]),
    widgets.HBox([lat_input, lon_input, goto_btn]),
    m, out,
]))


After you click **Show network**, the catchment polygon is available as `catchment_polygon` (and edges as `catchment_edges`) for the next cell, where we'll pull data into it.

## Assemble the structured spatial brief

Pulls everything **into the catchment polygon** and emits one structured object:
- **location** — country / region / city (reverse-geocoded from the clicked point via Nominatim)
- **population** — streamed from the GHS-POP COG, masked to the polygon (÷100 scaled)
- **road_length_m** — total reachable network length
- **poi_by_category** — OSM POI counts per amenity type within the polygon

This `brief` dict is exactly what feeds the Stage-2 persona'd LLM — all numbers computed here, none left to the model.

> Nominatim asks for ≤1 request/sec and a real user-agent — fine for a demo, don't hammer it.

### Population streamer (GHS-POP COG, areal-weighted)

Streams the **GHS-POP 2021** Cloud-Optimized GeoTIFF (100m, EPSG:4326) from OpenLandMap's public S3, reading only the pixels under the polygon via HTTP range requests. Each 100m cell contributes population × (cell area inside the catchment ÷ cell area) — **areal-weighted**, not all-or-nothing by centroid. Stored as UInt32, so ÷100 for real counts.

In [ ]:
""" # GHS-POP R2023A (JRC), 2021 annual layer re-hosted as COG by OpenLandMap, 100m, EPSG:4326.
# /vsicurl/ = stream over HTTP, fetch only needed tiles.
Areal-weighted GHS-POP within the polygon: each 100m cell contributes
population * (cell area inside polygon / cell area). No centroid all-or-nothing."""

import numpy as np
import rasterio
from rasterio.windows import from_bounds, Window
from shapely.geometry import box
import geopandas as gpd

# Public GHS-POP 2021 COG (100m, EPSG:4326). /vsicurl/ = stream over HTTP, fetch only needed tiles.
GHS_POP_COG = ('/vsicurl/https://s3.openlandmap.org/arco/'
               'pop.count_ghs.jrc_m_100m_s_20210101_20211231_go_epsg.4326_v20230620.tif')
GHS_SCALE = 100.0  # stored UInt32 -> divide by 100 for actual people

# allow GDAL to make range requests to public buckets
rasterio_env = rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR',
                            CPL_VSIL_CURL_USE_HEAD='NO')

def population_in_polygon(polygon_4326, cog=GHS_POP_COG, scale=GHS_SCALE, pad=2):
    with rasterio_env:
        with rasterio.open(cog) as src:
            b = polygon_4326.bounds
            win = from_bounds(*b, transform=src.transform)
            # pad window by a few cells so partial edge cells aren't dropped, clamp to raster
            col_off = max(0, int(np.floor(win.col_off)) - pad)
            row_off = max(0, int(np.floor(win.row_off)) - pad)
            width  = min(src.width  - col_off, int(np.ceil(win.width))  + 2 * pad)
            height = min(src.height - row_off, int(np.ceil(win.height)) + 2 * pad)
            win2 = Window(col_off, row_off, width, height)
            arr = src.read(1, window=win2).astype('float64')
            wt = src.window_transform(win2)
    arr[arr < 0] = 0
    # build cell polygons for the window and areal-weight against the catchment (in METRES)
    rows, cols = arr.shape
    polys, vals = [], []
    for r in range(rows):
        for c in range(cols):
            x0, y0 = wt * (c, r)
            x1, y1 = wt * (c + 1, r + 1)
            polys.append(box(min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1)))
            vals.append(arr[r, c])
    cells = gpd.GeoDataFrame({'v': vals}, geometry=polys, crs=4326)
    # reproject cells + catchment to a metric CRS so area/fraction are correct
    utm = cells.estimate_utm_crs()
    cells_m = cells.to_crs(utm)
    poly_m = gpd.GeoSeries([polygon_4326], crs=4326).to_crs(utm).iloc[0]
    cells_m['cell_area'] = cells_m.area
    inter = cells_m.copy()
    inter['geometry'] = cells_m.intersection(poly_m)
    inter = inter[~inter.is_empty]
    inter['frac'] = inter.area / inter['cell_area']
    return float((inter['v'] * inter['frac']).sum()) / scale

In [ ]:
# Run AFTER clicking 'Show network' so the catchment globals exist.
import json, requests
import geopandas as gpd
import osmnx as ox

assert 'catchment_polygon' in globals(), 'Run the map cell and click "Show network" first.'


def reverse_geocode(lat, lon):
    """Country / region / city for the clicked point via Nominatim reverse geocoding."""
    try:
        r = requests.get('https://nominatim.openstreetmap.org/reverse',
                         params={'lat': lat, 'lon': lon, 'format': 'jsonv2', 'zoom': 10},
                         headers={'User-Agent': 'network-catchment-demo/0.1'}, timeout=10)
        a = r.json().get('address', {})
        return {
            'country': a.get('country'),
            'region': a.get('state') or a.get('region') or a.get('province'),
            'city': a.get('city') or a.get('town') or a.get('village') or a.get('municipality'),
        }
    except Exception as e:
        return {'country': None, 'region': None, 'city': None, 'error': str(e)}


def _features_in_bbox(polygon_4326, tags):
    """Fetch OSM features in the polygon's BOUNDING BOX (cheap server-side query),
    to be filtered to the exact polygon locally. Sending the full complex catchment
    polygon to Overpass is what causes timeouts; a bbox query is fast and cacheable."""
    minx, miny, maxx, maxy = polygon_4326.bounds
    try:
        # osmnx 2.x preferred: bbox as (left, bottom, right, top)
        return ox.features_from_bbox((minx, miny, maxx, maxy), tags=tags)
    except TypeError:
        # fallback for point releases with keyword signature
        return ox.features_from_bbox(north=maxy, south=miny, east=maxx, west=minx, tags=tags)


def pois_in_polygon(polygon_4326):
    """POIs in the catchment (amenity + shop + leisure), counted per category.
    Fetched by bbox, then filtered to the polygon locally. Every feature (point OR
    polygon) is represented by an interior point, so polygon-mapped amenities count."""
    try:
        g = _features_in_bbox(polygon_4326, {'amenity': True, 'shop': True, 'leisure': True})
        if len(g) == 0:
            return {}
        def label(row):
            for k in ('amenity', 'shop', 'leisure'):
                if k in row and isinstance(row[k], str) and row[k]:
                    return row[k]
            return None
        g = g.copy()
        g['poi_cat'] = g.apply(label, axis=1)
        g = g[g['poi_cat'].notna()]
        # drop property-tags that aren't real destinations
        g = g[~g['poi_cat'].isin({'outdoor_seating', 'outdoor'})]
        # precise membership test, done locally
        reps = g.geometry.representative_point()
        g = g[reps.within(polygon_4326)]
        return g['poi_cat'].value_counts().to_dict()
    except Exception as e:
        print('POI query failed:', e)
        return {}


def buildings_in_polygon(polygon_4326, utm_crs, min_area_m2=15):
    """Centroid rule: a building belongs to the catchment if its centroid is inside.
    Fetched by bbox, filtered to the polygon locally. Filters out building:part
    fragments and tiny (<min_area_m2) features so the count reflects whole buildings.
    Returns (count, whole_footprint_m2, clipped_footprint_m2)."""
    try:
        g = _features_in_bbox(polygon_4326, {'building': True})
        g = g[g.geometry.type.isin(['Polygon', 'MultiPolygon'])]
        if 'building:part' in g.columns:
            g = g[g['building:part'].isna()]
        if len(g) == 0:
            print('buildings: query returned 0 polygon features')
            return 0, 0.0, 0.0
        gm = g.to_crs(utm_crs).reset_index(drop=True)
        gm = gm[gm.area >= min_area_m2]
        if len(gm) == 0:
            return 0, 0.0, 0.0
        poly_utm = gpd.GeoSeries([polygon_4326], crs=4326).to_crs(utm_crs).iloc[0]
        cents = gm.geometry.centroid
        inside = gm[cents.within(poly_utm).values]
        count = int(len(inside))
        whole_footprint = float(inside.area.sum())
        clipped = gm.geometry.intersection(poly_utm)
        clipped_footprint = float(clipped[~clipped.is_empty].area.sum())
        return count, whole_footprint, clipped_footprint
    except Exception as e:
        print('buildings query failed:', e)
        return 0, 0.0, 0.0


# --- assemble the structured spatial brief ---
lat, lon = click_point

# local metric CRS so all areas/lengths are in real metres
_poly_gs = gpd.GeoSeries([catchment_polygon], crs=4326)
utm_crs = _poly_gs.estimate_utm_crs()
catchment_area_m2 = float(_poly_gs.to_crs(utm_crs).area.iloc[0])

population = round(population_in_polygon(catchment_polygon))
poi_counts = pois_in_polygon(catchment_polygon)
bld_count, bld_footprint_whole, bld_footprint_clipped = buildings_in_polygon(catchment_polygon, utm_crs)
road_len = round(catchment_road_length_m)
area_km2 = catchment_area_m2 / 1e6

# --- derived indicators (computed here so the LLM never has to do arithmetic) ---
indicators = {
    'catchment_area_m2': round(catchment_area_m2),
    'population_density_per_km2': round(population / area_km2) if area_km2 else None,
    'building_count': bld_count,
    'building_footprint_m2': round(bld_footprint_whole),
    'building_coverage_ratio': round(bld_footprint_clipped / catchment_area_m2, 3) if catchment_area_m2 else None,
    'avg_building_footprint_m2': round(bld_footprint_whole / bld_count) if bld_count else None,
    'people_per_building': round(population / bld_count, 1) if bld_count else None,
    'road_density_m_per_km2': round(road_len / area_km2) if area_km2 else None,
    'poi_per_1000_residents': round(sum(poi_counts.values()) / population * 1000, 1) if population else None,
}

brief = {
    'retrieval_method': {
        'type': 'network catchment (walking-distance along streets), NOT a circular buffer',
        'network_distance_m': catchment_settings['distance_m'],
        'block_depth_m': catchment_settings['block_depth_m'],
        'note': 'values aggregated over the filled catchment: the area reachable on foot within network_distance_m, expanded off the streets by block_depth_m to fill blocks',
    },
    'location': reverse_geocode(lat, lon),
    'point': {'lat': round(lat, 6), 'lon': round(lon, 6)},
    'population': {
        'residents': population,
        'source': 'GHS-POP (modelled RESIDENTIAL population only)',
        'caveat': 'counts people who LIVE here, not daytime workers/visitors. In commercial '
                  'or mixed-use districts the resident count (and any per-resident ratio) can '
                  'be very low even where the area is busy and full of activity.',
    },
    'road_length_m': road_len,
    'poi_total': int(sum(poi_counts.values())),
    'poi_by_category': dict(sorted(poi_counts.items(), key=lambda kv: -kv[1])),
    'indicators': indicators,
}

print(json.dumps(brief, indent=2, ensure_ascii=False))

# --- retrieval sanity guard: never freeze a corrupt brief for the benchmark ---
if bld_count == 0 or brief['poi_total'] == 0:
    print('\n[WARNING] buildings or POIs came back empty - likely a failed/timed-out query.')
    print('          Re-run this cell before freezing this brief for the benchmark.')


# Stage 2 — query the location with a persona'd LLM

Pick a Qwen3 model from the dropdown and click **Load model** (switching frees the previous model's VRAM first). The context is **not** retrieved text chunks — it's the structured `brief` you computed, injected directly into the system prompt (pre-prompting). No vector store, no embedding.

The persona (a) gives the model a basic role, (b) tells it where it is, and (c) explains what each field of the brief means — so we can see whether the model actually reasons spatially over the numbers rather than just echoing them. Try the same question across model sizes to compare spatial reasoning.

## Hugging Face authentication (needed for gated models)

Llama and Gemma repos are **gated**: on each model's Hugging Face page you must request access and be approved (usually instant), then authenticate here with a token that has *read* permission. Qwen is not gated. Get a token at https://huggingface.co/settings/tokens

In [ ]:
!pip install -U 'transformers>=5.10.1' accelerate huggingface_hub bitsandbytes

In [ ]:
# --- Hugging Face authentication (needed for gated models: Llama, Gemma 3, Gemma 4) ---
from huggingface_hub import login, whoami

# Paste your token when the box appears (not saved into the notebook).
login()

# To hardcode instead (e.g. private Colab), comment out login() above and use:
# login(token='hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx')   # do NOT commit a real token

try:
    print('Authenticated as:', whoami()['name'])
except Exception as e:
    print('Not authenticated yet:', e)


In [ ]:
# --- Model picker: choose a model from three families, click Load ---
import torch, gc
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoProcessor, BitsAndBytesConfig
import ipywidgets as widgets
from IPython.display import display
import warnings
warnings.filterwarnings('ignore', message='.*_check_is_size.*', category=FutureWarning)

# label -> {'id': HF model id, 'thinking': True | False | None}
#   thinking=True/False -> passed as enable_thinking to apply_chat_template (Qwen3 hybrids, Gemma 4)
#   thinking=None       -> model has no thinking mode / no toggle (Llama 3.x, Gemma 3)
# Qwen3 (thinking) and (no-think) entries load the SAME weights; only the chat-template
# flag differs, so the pair is a controlled ablation of the reasoning trace.
MODELS = {
    # Qwen3 hybrid ladder -- thinking ON
    'Qwen3-1.7B (thinking)': {'id': 'Qwen/Qwen3-1.7B', 'thinking': True},
    'Qwen3-4B (thinking)':   {'id': 'Qwen/Qwen3-4B',   'thinking': True},
    'Qwen3-8B (thinking)':   {'id': 'Qwen/Qwen3-8B',   'thinking': True},
    'Qwen3-14B (thinking)':  {'id': 'Qwen/Qwen3-14B',  'thinking': True},

    # Qwen3 hybrid ladder -- thinking OFF (same weights, enable_thinking=False)
    'Qwen3-1.7B (no-think)': {'id': 'Qwen/Qwen3-1.7B', 'thinking': False},
    'Qwen3-4B (no-think)':   {'id': 'Qwen/Qwen3-4B',   'thinking': False},
    'Qwen3-8B (no-think)':   {'id': 'Qwen/Qwen3-8B',   'thinking': False},
    'Qwen3-14B (no-think)':  {'id': 'Qwen/Qwen3-14B',  'thinking': False},

    # NOTE: 'Qwen/Qwen3-4B-Instruct-2507' is a DIFFERENT checkpoint (later post-training,
    # non-thinking only, exists only at a few sizes). It breaks the same-weights ladder,
    # so it is excluded from the study. Uncomment only for side experiments:
    # 'Qwen3-4B-Instruct-2507 (extra)': {'id': 'Qwen/Qwen3-4B-Instruct-2507', 'thinking': None},

    # Llama 3.x (no thinking phase; full output is the answer)
    'Llama-3.2-1B-Instruct': {'id': 'meta-llama/Llama-3.2-1B-Instruct', 'thinking': None},
    'Llama-3.2-3B-Instruct': {'id': 'meta-llama/Llama-3.2-3B-Instruct', 'thinking': None},
    'Llama-3.1-8B-Instruct': {'id': 'meta-llama/Llama-3.1-8B-Instruct', 'thinking': None},

    # Gemma 3 (gated: requires HF auth; no thinking phase; no system role -> merged into user turn)
    'Gemma-3-1B-it':  {'id': 'google/gemma-3-1b-it',  'thinking': None},
    'Gemma-3-4B-it':  {'id': 'google/gemma-3-4b-it',  'thinking': None},
    'Gemma-3-12B-it': {'id': 'google/gemma-3-12b-it', 'thinking': None},

    # Gemma 4 (gated; DENSE 12B; built-in thinking mode, toggled via enable_thinking,
    # same-weights pair like the Qwen entries; needs transformers >= 5.10.1)
    'Gemma-4-12B-it (thinking)': {'id': 'google/gemma-4-12B-it', 'thinking': True},
    'Gemma-4-12B-it (no-think)': {'id': 'google/gemma-4-12B-it', 'thinking': False},
}

# --- vendor-recommended sampling (model cards / official docs, checked 2026-07) ---
# Qwen3 model card: thinking 0.6/0.95/20, non-thinking 0.7/0.8/20 (never greedy in thinking mode)
# Llama 3.x Instruct generation_config: 0.6/0.9 (no top_k specified)
# Gemma 3 & Gemma 4 (Google): 1.0/0.95/64 (same preset for both generations)
RECOMMENDED_SAMPLING = {
    ('qwen', True):    {'temperature': 0.6, 'top_p': 0.95, 'top_k': 20},
    ('qwen', False):   {'temperature': 0.7, 'top_p': 0.8,  'top_k': 20},
    ('llama', None):   {'temperature': 0.6, 'top_p': 0.9,  'top_k': None},
    ('gemma', None):   {'temperature': 1.0, 'top_p': 0.95, 'top_k': 64},
    ('gemma4', True):  {'temperature': 1.0, 'top_p': 0.95, 'top_k': 64},
    ('gemma4', False): {'temperature': 1.0, 'top_p': 0.95, 'top_k': 64},
}

def rec_sampling(fam=None, thinking='auto'):
    """Vendor-recommended sampling for the loaded (or given) family/thinking combo."""
    if fam is None:
        fam = globals().get('model_family', 'other')
    if thinking == 'auto':
        thinking = globals().get('model_thinking', None)
    return dict(RECOMMENDED_SAMPLING.get((fam, thinking),
                {'temperature': 0.7, 'top_p': 0.95, 'top_k': None}))

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# holds the currently loaded model/tokenizer + flags for the rest of the notebook
model = None
tokenizer = None       # a tokenizer (Qwen/Llama/Gemma3) OR a processor (Gemma 4)
model_family = None    # 'qwen' | 'llama' | 'gemma' | 'gemma4'
model_thinking = None  # True | False | None -- record this in run metadata!
model_label = None     # dropdown label of the loaded model, for logging

def _family_of(model_id):
    mid = model_id.lower()
    if 'qwen' in mid: return 'qwen'
    if 'llama' in mid: return 'llama'
    if 'gemma-4' in mid: return 'gemma4'   # thinking model; processor + parse_response
    if 'gemma' in mid: return 'gemma'      # Gemma 3: no thinking, no system role
    return 'other'

def _version_tuple(v):
    parts = []
    for p in v.split('.')[:3]:
        num = ''.join(ch for ch in p if ch.isdigit())
        parts.append(int(num) if num else 0)
    return tuple(parts)

model_dd = widgets.Dropdown(options=list(MODELS.keys()),
                            value='Qwen3-4B (thinking)',
                            description='Model:', layout={'width': '420px'})
load_btn = widgets.Button(description='Load model', button_style='primary')
load_out = widgets.Output()

def load_model(_=None):
    global model, tokenizer, model_family, model_thinking, model_label
    entry = MODELS[model_dd.value]
    model_id, thinking = entry['id'], entry['thinking']
    fam = _family_of(model_id)
    load_out.clear_output()
    with load_out:
        print(f'Loading {model_id} (thinking={thinking}) ...')
    if model is not None:
        del model
        model = None
        gc.collect()
        torch.cuda.empty_cache()

    if fam == 'gemma4':
        # Gemma 4 is multimodal: load via AutoProcessor + the multimodal model class
        # (per Google's docs; requires transformers >= 5.10.1).
        if _version_tuple(transformers.__version__) < (5, 10, 1):
            with load_out:
                print(f'ERROR: transformers {transformers.__version__} is too old for '
                      f'Gemma 4 (needs >= 5.10.1). Re-run the pip install cell and '
                      f'restart the runtime.')
            return
        try:
            from transformers import AutoModelForMultimodalLM as _Gemma4Class
        except ImportError:
            from transformers import AutoModelForImageTextToText as _Gemma4Class
        tokenizer = AutoProcessor.from_pretrained(model_id)
        model = _Gemma4Class.from_pretrained(
            model_id, quantization_config=bnb_config,
            device_map='auto', dtype=torch.bfloat16,
        )
    else:
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=bnb_config,
            device_map='auto', dtype=torch.bfloat16,
        )
    model.eval()
    model_family = fam
    model_thinking = thinking
    model_label = model_dd.value
    with load_out:
        print(f'Loaded {model_id} (family: {model_family}, thinking: {model_thinking}). '
              f'VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

load_btn.on_click(load_model)
display(widgets.VBox([widgets.HBox([model_dd, load_btn]), load_out]))


In [ ]:
# --- Persona + brief -> system prompt (pre-prompting) ---
import json

# Default persona shown in the widget; edit it live there.
DEFAULT_PERSONA = (
    'Input the persona here'
)

FIELD_GLOSSARY = (
    'The brief contains the following fields.\n'
    '- retrieval_method: how the area was defined. network_distance_m is the walking '
    'distance budget; block_depth_m is how far the catchment reaches off the streets to '
    'fill the blocks between them. The area is a network catchment along the streets, not a circle.\n'
    '- location / point: the country, region, and city the point falls in, and its latitude and longitude.\n'
    '- population: the number of residents living within the catchment, from GHS-POP, a modelled '
    'RESIDENTIAL population (2021 estimate) that counts people who live there, not daytime workers '
    'or visitors. In commercial or mixed-use areas this figure can be low even where the place is '
    'busy. Per-resident ratios such as poi_per_1000_residents are based on this residential figure.\n'
    '- road_length_m: the total length of walkable streets reachable within the catchment, in metres.\n'
    '- poi_total / poi_by_category: the count of points of interest (amenity, shop, and leisure '
    'features recorded in OpenStreetMap) found inside the catchment, broken down by category. '
    'The categories listed are those recorded within the catchment.\n'
    '- indicators: pre-computed metrics. catchment_area_m2 is the area of the filled catchment. '
    'population_density_per_km2 is residents per square kilometre. building_count is the number of '
    'buildings (excluding building parts and features under 15 m2); building_footprint_m2 is the '
    'summed footprint of those buildings, whole; avg_building_footprint_m2 is their mean footprint. '
    'building_coverage_ratio is the built footprint CLIPPED to the catchment boundary divided by '
    'catchment area, so it can differ slightly from building_footprint_m2 divided by the area '
    '(which sums whole footprints of buildings centred inside). people_per_building is residents '
    'divided by buildings. road_density_m_per_km2 is road length per square kilometre. '
    'poi_per_1000_residents is points of interest divided by residents, times 1000 (for example, '
    'a value of 100 means 100 points of interest for every 1000 residents).'
)

# The glossary actually injected into prompts. Edited live via the glossary widget below;
# FIELD_GLOSSARY above is only the preloaded default.
CURRENT_GLOSSARY = FIELD_GLOSSARY

def build_system_prompt(brief, persona):
    loc = brief.get('location', {})
    place = ', '.join(v for v in [loc.get('city'), loc.get('region'), loc.get('country')] if v) or 'an unspecified location'
    return (
        f'{persona}\n\n'
        f'You are reasoning about a point in {place}.\n\n'
        f"{globals().get('CURRENT_GLOSSARY', FIELD_GLOSSARY)}\n\n"
        f'SPATIAL BRIEF:\n{json.dumps(brief, indent=2, ensure_ascii=False)}'
    )

def _merge_system_into_first_user(history, question, sys_prompt):
    """Fold the system prompt into the FIRST user message (for templates with no system role)."""
    messages = []
    injected = False
    for m in (list(history) + [{'role': 'user', 'content': question}]):
        if m['role'] == 'user' and not injected:
            messages.append({'role': 'user', 'content': sys_prompt + '\n\n' + m['content']})
            injected = True
        else:
            messages.append({'role': m['role'], 'content': m['content']})
    return messages

def ask_location(question, brief, persona, history,
                 max_new_tokens=1500, temperature=0.9, top_p=None, top_k=None, seed=None):
    """Generate an answer by sampling at `temperature`.
    Each model runs in its native chat-template mode:
      - Qwen3 hybrids: enable_thinking (from the dropdown pair) toggles the <think> block
        on the SAME weights. When ON, the block is stripped below; only the final answer
        is scored. When OFF, no block is emitted.
      - Llama 3.x has no thinking phase; its full output is the answer.
      - Gemma 3 has no thinking phase AND no system role, so the system prompt is merged
        into the first user turn.
      - Gemma 4 has a built-in thinking mode, also toggled via enable_thinking. Output is
        decoded with skip_special_tokens=False and split into thinking/answer with the
        processor's parse_response(); only the answer is kept. History stores answers
        only, so prior-turn thoughts never re-enter the context (as Google's docs require).
    Long-thinking models may need a higher max_new_tokens to avoid truncation.
    Pass an integer `seed` for a reproducible sample (vary the seed across the 10 runs).
    Record model_label / model_family / model_thinking in the run metadata for every sample.
    history = list of prior {'role','content'} turns, for memory.
    """
    if model is None or tokenizer is None:
        return '[No model loaded \u2014 pick one in the dropdown above and click "Load model" first.]'

    sys_prompt = build_system_prompt(brief, persona)
    fam = globals().get('model_family', 'other')
    thinking = globals().get('model_thinking', None)

    if fam == 'gemma':
        # Gemma 3's chat template rejects a system role.
        messages = _merge_system_into_first_user(history, question, sys_prompt)
    else:
        messages = [{'role': 'system', 'content': sys_prompt}]
        messages += history
        messages += [{'role': 'user', 'content': question}]

    # enable_thinking is only meaningful for hybrid thinking models (Qwen3, Gemma 4)
    template_kwargs = {}
    if fam in ('qwen', 'gemma4') and thinking is not None:
        template_kwargs['enable_thinking'] = thinking

    try:
        text = tokenizer.apply_chat_template(messages, tokenize=False,
                                             add_generation_prompt=True, **template_kwargs)
    except Exception:
        if fam == 'gemma4':
            # defensive fallback in case this template rejects a system role like Gemma 3
            messages = _merge_system_into_first_user(history, question, sys_prompt)
            text = tokenizer.apply_chat_template(messages, tokenize=False,
                                                 add_generation_prompt=True, **template_kwargs)
        else:
            raise

    # keyword `text=` works for both tokenizers and processors (a processor's first
    # positional argument is images, so positional calling would break on Gemma 4)
    inputs = tokenizer(text=text, return_tensors='pt').to(model.device)

    # the underlying tokenizer (a Gemma 4 processor wraps one)
    tok = getattr(tokenizer, 'tokenizer', tokenizer)

    gen_kwargs = {}
    if fam == 'gemma4':
        # let Gemma 4 use its own generation_config stop tokens (its templates use
        # turn/channel special tokens, not plain EOS)
        gen_kwargs['pad_token_id'] = tok.eos_token_id
    else:
        # stop tokens: standard EOS, plus Gemma 3's <end_of_turn> (it does not stop on plain EOS)
        eos_ids = [tok.eos_token_id]
        if fam == 'gemma':
            eot = tok.convert_tokens_to_ids('<end_of_turn>')
            if isinstance(eot, int) and eot >= 0:
                eos_ids.append(eot)
        gen_kwargs['eos_token_id'] = eos_ids
        gen_kwargs['pad_token_id'] = tok.eos_token_id

    if seed is not None:
        torch.manual_seed(int(seed))
    with torch.no_grad():
        if top_p is not None:
            gen_kwargs['top_p'] = float(top_p)
        if top_k is not None:
            gen_kwargs['top_k'] = int(top_k)
        gen = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=True, temperature=float(temperature),
                             **gen_kwargs)

    gen_ids = gen[0][inputs['input_ids'].shape[1]:]

    # truncation is detected by hitting the token cap, NOT by the absence of a
    # closing think tag (non-thinking runs never emit one, and that test would
    # delete their answers).
    truncated = (len(gen_ids) >= max_new_tokens)

    if fam == 'gemma4':
        # Gemma 4 turn format: <|channel>thought\n{thinking}<channel|>{answer}<turn|>
        # Decode WITH special tokens (the <channel|> separator is itself a special
        # token; skip_special_tokens=True would delete the only thinking/answer
        # boundary). Extraction chain: parse_response -> manual channel split.
        import re as _re
        raw = tokenizer.decode(gen_ids, skip_special_tokens=False)

        def _strip_ctrl(s):
            # remove Gemma 4 control tokens (<|turn>model, <turn|>, <|channel>thought,
            # <channel|>, <|think|>, <bos>/<eos>/<pad>) from a text fragment
            s = _re.sub(r'<\|[^<>]{1,24}\|>', '', s)      # <|think|>, <|image|>...
            s = _re.sub(r'<\|[^<>]{1,24}>\w*', '', s)     # <|channel>thought, <|turn>model
            s = _re.sub(r'<[^<>]{1,24}\|>', '', s)         # <channel|>, <turn|>
            s = _re.sub(r'</?(bos|eos|pad)>', '', s)
            return s.strip()

        answer = None
        try:
            # new-style response_template parsers need the rendered prompt as prefix=
            # (the template pre-writes the thought-channel opener inside the
            # assistant turn, and the parser must see it to segment correctly)
            try:
                parsed = tokenizer.parse_response(raw, prefix=text)
            except TypeError:
                parsed = tokenizer.parse_response(raw)   # older signature
            answer = (parsed.get('content') or '').strip()
        except Exception as e:
            print(f'[gemma4] parse_response unavailable/failed ({type(e).__name__}: {e}); '
                  f'using manual channel split')
        if not answer:
            # manual split: the final answer is everything after the LAST <channel|>
            # (thinking ON: prompt or model opens the thought channel, model closes it
            # with <channel|> before the answer). If no separator was generated
            # (thinking OFF: template pre-closes an empty channel inside the prompt),
            # the whole generation is the answer.
            tail = raw.rsplit('<channel|>', 1)[-1] if '<channel|>' in raw else raw
            answer = _strip_ctrl(tail)
        if not answer:
            answer = '[EMPTY ANSWER after channel split; raw kept for inspection]\n' + _strip_ctrl(raw)
    else:
        full = tok.decode(gen_ids, skip_special_tokens=True)
        # strip the thinking block ONLY when one can exist (Qwen with thinking ON).
        if fam == 'qwen' and '</think>' in full:
            answer = full.split('</think>')[-1].strip()
        else:
            answer = full.strip()   # Llama / Gemma 3 / Qwen no-think: the full output IS the answer

    # surface truncation visibly but KEEP the text so it can be inspected.
    if truncated:
        if not answer:
            answer = ('[TRUNCATED: hit the ' + str(max_new_tokens) +
                      '-token limit before producing any answer (model likely still thinking). '
                      'Raise max tokens and rerun this seed.]')
        else:
            answer = ('[TRUNCATED: hit the ' + str(max_new_tokens) +
                      '-token limit; the answer below is incomplete]\n' + answer)
    return answer

# --- Interactive query widget: persona + memory + sampling controls ---
import ipywidgets as widgets
from IPython.display import display

assert 'brief' in globals(), 'Run the brief cell first so `brief` exists.'

glossary_box = widgets.Textarea(
    value=FIELD_GLOSSARY,
    description='Glossary:',
    layout={'width': '100%', 'height': '180px'},
    style={'description_width': 'initial'},
)
apply_gloss_btn = widgets.Button(description='Inject glossary', button_style='info')
gloss_status = widgets.HTML(value='<i>glossary: preloaded default</i>')

def on_apply_glossary(_=None):
    global CURRENT_GLOSSARY
    CURRENT_GLOSSARY = glossary_box.value
    import hashlib
    h = hashlib.sha256(CURRENT_GLOSSARY.encode('utf-8')).hexdigest()[:12]
    tag = 'default' if CURRENT_GLOSSARY == FIELD_GLOSSARY else 'CUSTOM'
    gloss_status.value = f'<i>glossary injected ({tag}, sha256:{h}, {len(CURRENT_GLOSSARY)} chars)</i>'

apply_gloss_btn.on_click(on_apply_glossary)

persona_box = widgets.Textarea(
    value=DEFAULT_PERSONA,
    description='Persona:',
    layout={'width': '100%', 'height': '120px'},
    style={'description_width': 'initial'},
)

# --- sampling controls (always sampling; vary seed across the 10 runs per test) ---
temp_slider = widgets.FloatSlider(value=0.9, min=0.0, max=2.0, step=0.05,
                                  description='Temperature:', readout_format='.2f',
                                  style={'description_width': 'initial'},
                                  layout={'width': '320px'})
maxtok_slider = widgets.IntSlider(value=10000, min=100, max=16000, step=100,
                                  description='Max tokens:',
                                  style={'description_width': 'initial'},
                                  layout={'width': '320px'})
seed_input = widgets.IntText(value=1, description='Seed:',
                             layout={'width': '160px'},
                             style={'description_width': 'initial'})
topp_slider = widgets.FloatSlider(value=0.95, min=0.05, max=1.0, step=0.01,
                                  description='Top-p:', readout_format='.2f',
                                  style={'description_width': 'initial'},
                                  layout={'width': '280px'})
topk_input = widgets.IntText(value=0, description='Top-k (0=off):',
                             layout={'width': '190px'},
                             style={'description_width': 'initial'})
rec_btn = widgets.Button(description='Use recommended', button_style='info',
                         tooltip='Set temp/top-p/top-k to the vendor recommendation for the loaded model')

def on_rec(_=None):
    r = rec_sampling()
    temp_slider.value = r['temperature']
    topp_slider.value = r['top_p']
    topk_input.value = r['top_k'] or 0

rec_btn.on_click(on_rec)

entry = widgets.Text(placeholder='Ask about this location\u2026', layout={'width': '100%'})
send_btn = widgets.Button(description='Ask', button_style='primary')
reset_btn = widgets.Button(description='Reset memory', button_style='warning')
out = widgets.Output(layout={'border': '1px solid #ccc', 'height': '360px',
                             'overflow': 'auto', 'padding': '8px'})

history = []

def on_ask(_=None):
    q = entry.value.strip()
    if not q:
        return
    entry.value = ''
    with out:
        lbl = globals().get('model_label', '?')
        thk = globals().get('model_thinking', None)
        print(f'Q: {q}  [model={lbl}, thinking={thk}, temp={temp_slider.value:.2f}, '
              f'top_p={topp_slider.value:.2f}, top_k={topk_input.value or None}, '
              f'seed={seed_input.value}]')
        print('...thinking...')
    ans = ask_location(q, brief, persona_box.value, history,
                       max_new_tokens=maxtok_slider.value,
                       temperature=temp_slider.value,
                       top_p=topp_slider.value,
                       top_k=(topk_input.value or None),
                       seed=seed_input.value)
    history.append({'role': 'user', 'content': q})
    history.append({'role': 'assistant', 'content': ans})
    with out:
        print(f'A: {ans}\n')

def on_reset(_=None):
    history.clear()
    out.clear_output()
    with out:
        print('[memory cleared \u2014 starting over]\n')

send_btn.on_click(on_ask)
reset_btn.on_click(on_reset)
entry.on_submit(on_ask)

controls = widgets.HBox([temp_slider, topp_slider, topk_input, rec_btn])
controls2 = widgets.HBox([maxtok_slider, seed_input])

display(widgets.VBox([
    glossary_box,
    widgets.HBox([apply_gloss_btn, gloss_status]),
    persona_box,
    controls,
    controls2,
    out,
    widgets.HBox([entry, send_btn, reset_btn]),
]))


# Stage 3 — Benchmark runner (10 seeds × 3 questions, saved to Drive)

Load a model above, click **Inject glossary** if you edited it, paste the frozen brief + persona + Q1–Q3, then run. Sampling defaults to the vendor recommendation for the loaded model (uncheck to set manually). One JSONL per model config in `MyDrive/nscr_benchmark/`; every record stores family, thinking flag, sampling, and glossary/persona hashes.

On non-Colab hosts (e.g. SaladCloud) Google Drive mounting is unavailable: records are written locally and, if `rclone` is installed with a `gdrive` remote (env `RCLONE_REMOTE`, default `gdrive:nscr_benchmark`), pushed to Drive after every record. Reruns skip records that already exist (resume after interruption); partially-completed seeds rebuild their conversation history from the saved answers.

In [ ]:
# --- Benchmark runner: 10 seeds x (Q1,Q2,Q3) with memory, saved per-model to Drive ---
import os, json, time, datetime, hashlib, re, shutil, subprocess
import ipywidgets as widgets
from IPython.display import display

# --- mount Google Drive (idempotent) ---
DRIVE_DIR = '/content/drive/MyDrive/nscr_benchmark'
try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    os.makedirs(DRIVE_DIR, exist_ok=True)
    _drive_ok = True
except Exception as e:
    # fallback to local dir if not on Colab
    DRIVE_DIR = os.path.abspath('./nscr_benchmark')
    os.makedirs(DRIVE_DIR, exist_ok=True)
    _drive_ok = False
    print('Google Drive not available, saving locally to', DRIVE_DIR, '(', e, ')')

# --- paste fields ---
case_box   = widgets.Text(value='paris_400m', description='Case name:',
                          layout={'width': '420px'}, style={'description_width': 'initial'})
persona_ta = widgets.Textarea(value=DEFAULT_PERSONA, description='Persona:',
                              layout={'width': '100%', 'height': '110px'},
                              style={'description_width': 'initial'})
brief_ta   = widgets.Textarea(value='', placeholder='Paste the frozen brief JSON here...',
                              description='Brief JSON:', layout={'width': '100%', 'height': '160px'},
                              style={'description_width': 'initial'})
q1_ta = widgets.Textarea(value='', description='Q1:', layout={'width': '100%', 'height': '60px'},
                         style={'description_width': 'initial'})
q2_ta = widgets.Textarea(value='', description='Q2:', layout={'width': '100%', 'height': '60px'},
                         style={'description_width': 'initial'})
q3_ta = widgets.Textarea(value='', description='Q3:', layout={'width': '100%', 'height': '60px'},
                         style={'description_width': 'initial'})

seeds_box  = widgets.Text(value='1,2,3,4,5,6,7,8,9,10', description='Seeds:',
                          layout={'width': '420px'}, style={'description_width': 'initial'})

# --- sampling: vendor-recommended per loaded model (default), or manual override ---
use_rec_chk = widgets.Checkbox(value=True, indent=False,
                               description='Use vendor-recommended sampling for the loaded model')
temp_box   = widgets.FloatText(value=0.7, description='Temp:', layout={'width': '160px'})
topp_box   = widgets.FloatText(value=0.95, description='Top-p:', layout={'width': '170px'})
topk_box   = widgets.IntText(value=0, description='Top-k (0=off):', layout={'width': '190px'},
                             style={'description_width': 'initial'})
maxtok_box = widgets.IntText(value=10000, description='Max tokens:', layout={'width': '200px'},
                             style={'description_width': 'initial'})

run_btn = widgets.Button(description='Run 10 seeds', button_style='success')
run_out = widgets.Output(layout={'border': '1px solid #ccc', 'height': '300px',
                                 'overflow': 'auto', 'padding': '8px'})

def _model_label():
    # reverse-lookup the dropdown label for the loaded model id
    lbl = globals().get('model_label', None)
    if lbl:
        return lbl
    try:
        return model_dd.value
    except Exception:
        return 'unknown_model'

def _safe_name(label):
    return re.sub(r'[^A-Za-z0-9._-]+', '_', label).strip('_')

def _sha(s):
    return hashlib.sha256(s.encode('utf-8')).hexdigest()[:12]

# --- interruption safety (SaladCloud / any non-Colab host) ---
# If rclone is installed and RCLONE_REMOTE is set (e.g. 'gdrive:nscr_benchmark'),
# every record is pushed to Drive right after it is written. Combined with the
# resume-skip below, an interrupted container can be restarted and will continue
# exactly where it stopped (pull existing files down first: see _pull_existing).
# A truncated answer this long is treated as degeneration (runaway repetition):
# it is recorded, but the seed's remaining questions are aborted so garbage never
# enters the conversation history (and never OOMs the next turn).
DEGEN_CHARS = 8000

RCLONE_REMOTE = os.environ.get('RCLONE_REMOTE', 'gdrive:nscr_benchmark')
_rclone = shutil.which('rclone')

def _sync_up(path):
    if _rclone and not _drive_ok:
        subprocess.run([_rclone, 'copy', path, RCLONE_REMOTE, '--quiet'],
                       check=False, timeout=120)

def _pull_existing(fname):
    if _rclone and not _drive_ok:
        subprocess.run([_rclone, 'copy', f'{RCLONE_REMOTE}/{fname}', DRIVE_DIR, '--quiet'],
                       check=False, timeout=120)

def _done_keys(out_path):
    done = set()
    if os.path.exists(out_path):
        with open(out_path, encoding='utf-8') as f:
            for line in f:
                try:
                    r = json.loads(line)
                    done.add((r['model_label'], r['case'], r['seed'], r['q_index']))
                except Exception:
                    continue
    return done

def on_run(_=None):
    run_out.clear_output()
    with run_out:
        # --- validate ---
        if model is None or tokenizer is None:
            print('No model loaded. Pick one above and click Load model first.'); return
        try:
            brief_obj = json.loads(brief_ta.value.strip())
        except Exception as e:
            print('Brief JSON is not valid:', e); return
        persona = persona_ta.value.strip()
        qs = [q1_ta.value.strip(), q2_ta.value.strip(), q3_ta.value.strip()]
        if not all(qs):
            print('All three questions (Q1, Q2, Q3) must be filled.'); return
        if 'Input the persona here' in persona or not persona:
            print('Persona looks empty/placeholder, paste the real persona first.'); return
        try:
            seeds = [int(x) for x in seeds_box.value.replace(' ', '').split(',') if x != '']
        except Exception as e:
            print('Seeds must be comma-separated integers:', e); return

        model_lbl = _model_label()
        fam = globals().get('model_family', 'other')
        thinking = globals().get('model_thinking', None)
        case = case_box.value.strip()
        out_path = os.path.join(DRIVE_DIR, f'{_safe_name(model_lbl)}.jsonl')
        _pull_existing(os.path.basename(out_path))
        done = _done_keys(out_path)
        if done:
            print(f'Resume: {len(done)} records already present, matching (seed, Q) pairs will be skipped.')

        # --- resolve sampling ---
        if use_rec_chk.value:
            r = rec_sampling(fam, thinking)
            temperature, top_p, top_k = r['temperature'], r['top_p'], r['top_k']
            sampling_source = 'vendor_recommended'
        else:
            temperature = float(temp_box.value)
            top_p = float(topp_box.value)
            top_k = int(topk_box.value) or None
            sampling_source = 'manual'

        gloss = globals().get('CURRENT_GLOSSARY', FIELD_GLOSSARY)
        print(f'Model: {model_lbl} (family {fam}, thinking {thinking})  |  case: {case}  |  seeds: {seeds}')
        print(f'Sampling [{sampling_source}]: temp={temperature}, top_p={top_p}, top_k={top_k}')
        print(f'Glossary sha256:{_sha(gloss)}  |  persona sha256:{_sha(persona)}')
        print(f'Writing to: {out_path}\n')

        # preload existing answers so a partially-done seed can rebuild its history
        prev_answers = {}
        if os.path.exists(out_path):
            with open(out_path, encoding='utf-8') as f:
                for line in f:
                    try:
                        r = json.loads(line)
                        prev_answers[(r['model_label'], r['case'], r['seed'], r['q_index'])] = r['answer']
                    except Exception:
                        continue

        n_written = 0
        for seed in seeds:
            history = []                       # reset memory between seeds
            for qi, q in enumerate(qs, start=1):
                key = (model_lbl, case, seed, qi)
                if key in done:
                    ans_prev = prev_answers.get(key, '')
                    history.append({'role': 'user', 'content': q})
                    history.append({'role': 'assistant', 'content': ans_prev})
                    print(f'  seed {seed} Q{qi}: skipped (already done)')
                    continue
                t0 = time.time()
                ans = ask_location(q, brief_obj, persona, history,
                                   max_new_tokens=maxtok_box.value,
                                   temperature=temperature, top_p=top_p, top_k=top_k,
                                   seed=seed)
                dt = time.time() - t0
                history.append({'role': 'user', 'content': q})
                history.append({'role': 'assistant', 'content': ans})
                degenerate = ans.startswith('[TRUNCATED') and len(ans) >= DEGEN_CHARS
                rec = {
                    'timestamp': datetime.datetime.now().isoformat(timespec='seconds'),
                    'model_label': model_lbl,
                    'model_family': fam,
                    'model_thinking': thinking,
                    'case': case,
                    'seed': seed,
                    'q_index': qi,
                    'question': q,
                    'temperature': temperature,
                    'top_p': top_p,
                    'top_k': top_k,
                    'sampling_source': sampling_source,
                    'max_tokens': maxtok_box.value,
                    'glossary_sha256': _sha(gloss),
                    'persona_sha256': _sha(persona),
                    'gen_seconds': round(dt, 1),
                    'truncated': ans.startswith('[TRUNCATED'),
                    'degenerate': degenerate,
                    'answer': ans,
                }
                with open(out_path, 'a', encoding='utf-8') as f:
                    f.write(json.dumps(rec, ensure_ascii=False) + '\n')
                _sync_up(out_path)
                n_written += 1
                flag = ' [TRUNCATED]' if rec['truncated'] else ''
                print(f'  seed {seed} Q{qi}: {dt:.0f}s, {len(ans)} chars{flag}')
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass
                if degenerate:
                    # write explicit aborted records for the remaining questions so
                    # resume-skip does not attempt them later, then move to next seed
                    for qj in range(qi + 1, len(qs) + 1):
                        arec = dict(rec)
                        arec.update({'q_index': qj, 'question': qs[qj - 1],
                                     'gen_seconds': 0.0, 'truncated': False,
                                     'degenerate': False, 'answer': '',
                                     'aborted': 'seed aborted after degenerate answer'})
                        with open(out_path, 'a', encoding='utf-8') as f:
                            f.write(json.dumps(arec, ensure_ascii=False) + '\n')
                    _sync_up(out_path)
                    print(f'  seed {seed}: DEGENERATE answer at Q{qi}; remaining questions aborted')
                    break
            print(f'-- seed {seed} done --')
        print(f'\nFinished: {n_written} answers appended to {out_path}')

run_btn.on_click(on_run)

display(widgets.VBox([
    widgets.HBox([case_box, seeds_box]),
    use_rec_chk,
    widgets.HBox([temp_box, topp_box, topk_box, maxtok_box]),
    persona_ta, brief_ta, q1_ta, q2_ta, q3_ta,
    run_btn, run_out,
]))


## Optional: Copy / paste a brief to skip Stage 1

**Copy:** the box shows the current `brief` as JSON. Click into it, select all (Ctrl+A / Cmd+A) and copy (Ctrl+C / Cmd+C) to keep it anywhere (a file, the paper, an email).

**Paste:** drop a saved brief JSON into the second box and click **Load pasted brief** to set `brief`, then go straight to Stage 2. Works the same on Colab and locally, and survives session close because the brief lives in your clipboard, not the disk.

In [ ]:
# --- Copy / paste a brief (no files, works everywhere) ---
import json
import ipywidgets as widgets
from IPython.display import display

# ---- COPY: show current brief as JSON to select + copy ----
copy_box = widgets.Textarea(
    value=json.dumps(brief, indent=2, ensure_ascii=False) if 'brief' in globals() else '(no brief yet - run the brief cell first)',
    description='Copy:',
    layout={'width': '100%', 'height': '180px'},
    style={'description_width': 'initial'},
)
refresh_copy = widgets.Button(description='Refresh from current brief')

def on_refresh_copy(_):
    copy_box.value = (json.dumps(brief, indent=2, ensure_ascii=False)
                      if 'brief' in globals() else '(no brief yet)')
refresh_copy.on_click(on_refresh_copy)

# ---- PASTE: paste JSON here, load it into `brief` ----
paste_box = widgets.Textarea(
    value='',
    placeholder='Paste a saved brief JSON here, then click "Load pasted brief"...',
    description='Paste:',
    layout={'width': '100%', 'height': '180px'},
    style={'description_width': 'initial'},
)
load_btn = widgets.Button(description='Load pasted brief', button_style='info')
load_out = widgets.Output()

def on_load(_):
    load_out.clear_output()
    with load_out:
        txt = paste_box.value.strip()
        if not txt:
            print('Nothing pasted.')
            return
        try:
            loaded = json.loads(txt)
        except json.JSONDecodeError as e:
            print('Not valid JSON:', e)
            return
        globals()['brief'] = loaded
        print('Loaded. `brief` is now set - go straight to Stage 2 (load a model and ask).')
load_btn.on_click(on_load)

display(widgets.VBox([
    widgets.HBox([refresh_copy]),
    copy_box,
    paste_box,
    widgets.HBox([load_btn]),
    load_out,
]))
